In [16]:
import numpy as np
import random

In [17]:
# Define the distance matrix (distances between cities)
# Replace this with your distance matrix or generate one based on your problem
# Example distance matrix (replace this with your actual data)
distance_matrix = np.array([
[0, 10, 15, 20],
[10, 0, 35, 25],
[15, 35, 0, 30],
[20, 25, 30, 0]
])

In [18]:
# Parameters for Ant Colony Optimization
num_ants = 10
num_iterations = 50
evaporation_rate = 0.5
pheromone_constant = 1.0
heuristic_constant = 1.0

In [20]:
with np.errstate(divide='ignore'):
    visibility = 1 / distance_matrix
    visibility[visibility == np.inf] = 0  # Replace infinity with zero

In [25]:
import numpy as np
import random

# Correcting the ACO algorithm loop
for iteration in range(num_iterations):
    ant_routes = []
    for ant in range(num_ants):
        current_city = random.randint(0, num_cities - 1)
        visited_cities = [current_city]
        route = [current_city]
        
        while len(visited_cities) < num_cities:
            candidate_cities = []
            candidate_probs = []
            
            for city in range(num_cities):
                if city not in visited_cities:
                    # Calculate weight for each city
                    p = (pheromone[current_city][city] ** pheromone_constant) * \
                        (visibility[current_city][city] ** heuristic_constant)
                    candidate_cities.append(city)
                    candidate_probs.append(p)
            
            # STOCHASTIC SELECTION (Roulette Wheel) 
            # Instead of taking the max, ants choose based on probability
            prob_sum = sum(candidate_probs)
            norm_probs = [p / prob_sum for p in candidate_probs]
            selected_city = np.random.choice(candidate_cities, p=norm_probs)
            
            route.append(selected_city)
            visited_cities.append(selected_city)
            current_city = selected_city
        
        ant_routes.append(route)

    # 2. UPDATE PHEROMONE LEVELS (Outside the ant loop)
    delta_pheromone = np.zeros((num_cities, num_cities))
    for route in ant_routes:
        # Calculate total distance of this specific route
        route_distance = sum(distance_matrix[route[i]][route[i+1]] for i in range(len(route)-1))
        # Add returning to start city distance
        route_distance += distance_matrix[route[-1]][route[0]]
        
        # Contribution is 1 / Total Distance (prevents divide by zero)
        contribution = 1.0 / route_distance if route_distance > 0 else 0
        
        for i in range(len(route) - 1):
            city_a, city_b = route[i], route[i+1]
            delta_pheromone[city_a][city_b] += contribution
            delta_pheromone[city_b][city_a] += contribution
            
    # Apply evaporation and add new pheromones
    pheromone = (1 - evaporation_rate) * pheromone + delta_pheromone

In [34]:
# 1. Calculate the total distance for every ant's route in the current iteration
route_distances = []
for route in ant_routes:
    # Sum the distances between consecutive cities in the route
    distance = sum(distance_matrix[route[i]][route[i+1]] for i in range(len(route)-1))
    # Add the distance to return to the starting city (to complete the loop)
    distance += distance_matrix[route[-1]][route[0]]
    route_distances.append(distance)

# 2. Find the index of the MINIMUM distance (not max)
best_route_index = np.argmin(route_distances)

# 3. Retrieve the best route and its corresponding distance
best_route = ant_routes[best_route_index]
shortest_distance = route_distances[best_route_index]

print(f"Shortest Distance: {shortest_distance}")
print(f"Best Route: {best_route}")

Shortest Distance: 80
Best Route: [2, 0, 1, 3]
